# ⚖️ Co-piloto Jurídico: recorrido por el proyecto (v11.2)

**Para quién:** la defensa penal que prepara o contesta un recurso de apelación ante las Salas Penales del Supremo
Tribunal de Justicia de Jalisco (STJ).

**Qué hace:** recibe la descripción del caso y devuelve las sentencias del propio tribunal más parecidas, cada una con
su ficha (delito, acto impugnado, agravios y cómo los calificó la Sala, sentido del fallo, fundamento legal) y una
síntesis de los precedentes favorables y desfavorables. No predice resultados ni redacta agravios.

**Cómo leer este notebook:** cada sección es una fase del proyecto y llama al código del paquete `copiloto_juridico`
(el mismo que usan la app y la evaluación), sin copiarlo. Así lo que se ve aquí es lo que corre en la app.

| Fase | Qué hace | Código |
|---|---|---|
| 0. Datos | Origen de las sentencias y descarga | `datos/descarga.py`, `evaluacion/descargas/` |
| 1. Procesamiento | PDF → texto limpio, validación de descargas, corpus | `datos/procesamiento.py`, `datos/corpus.py` |
| 2. Índice | Fragmentos + embeddings + BM25 | `datos/indice.py`, `vectorstore.py` |
| 3. Co-piloto | Los 5 pasos, con el delito opcional | `pipeline.py`, `clasificador.py`, `fichas.py` |
| 4. Evaluación | Referencia automática, jueces, corridas | `evaluacion/` |

Se ejecuta con el entorno del proyecto (`pip install -e .`). La fase 3 necesita Ollama con `llama3.1:8b`; el resto no
usa el LLM.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

from copiloto_juridico import config

os.chdir(config.ROOT_DIR)  # las rutas del notebook son relativas a la raíz del repositorio

pd.set_option("display.max_colwidth", 120)
llm = config.LLMSettings()
print(f"LLM: {llm.model} · temperatura {llm.temperature} · semilla {llm.seed} · ventana {llm.num_ctx:,} tokens")
print(f"Embeddings: {config.EMBEDDING_MODEL_NAME} · máximo {config.MAX_SENTENCIAS_POR_TIPO} sentencias por tipo")

LLM: llama3.1:8b · temperatura 0.0 · semilla 42 · ventana 8,192 tokens
Embeddings: intfloat/multilingual-e5-base · máximo 30 sentencias por tipo


## Fase 0. Datos: de dónde salen las sentencias

Las sentencias son públicas y vienen del portal del STJ Jalisco. Al principio se descargaban con su API
(`python -m copiloto_juridico.cli descargar`, en `datos/descarga.py`): descarga balanceada, máximo 30 por tipo
(materia + delito) y validación de cada PDF.

**Cambio en octubre de 2026:** la API empezó a exigir reCAPTCHA. El descargador lo detecta y **se detiene**; el
proyecto no intenta evadir esa verificación. Desde entonces el corpus se amplía con **descargas a mano** desde el
portal, guiadas por tres listas:

| Lista | Para qué |
|---|---|
| `evaluacion/descargas/1_incompletas.csv` | Sentencias que se procesaron con una versión anterior que omitía páginas: volver a bajarlas completas |
| `evaluacion/descargas/2_nuevas.csv` | Tipos penales con menos de 30 sentencias en el corpus: sentencias nuevas |
| `evaluacion/descargas/3_volver_a_bajar.csv` | Descargas que llegaron con el texto de otra resolución |

Cada PDF se guarda con el nombre que identifica la publicación en el portal: fecha y hora de publicación al segundo,
materia y delito (`2025-09-10-114715_penal_robo-calificado.pdf`). Ese nombre es la clave para no confundir dos
resoluciones del mismo toca o del mismo día.

In [2]:
from copiloto_juridico.datos.procesamiento import parse_metadata_from_filename

for nombre in ["1_incompletas", "2_nuevas", "3_volver_a_bajar"]:
    p = Path(f"evaluacion/descargas/{nombre}.csv")
    if p.exists():
        df = pd.read_csv(p, dtype=str)
        print(f"{nombre}.csv: {len(df):,} filas · columnas: {', '.join(df.columns)}")

parse_metadata_from_filename("2025-09-10-114715_penal_robo-calificado.pdf")

1_incompletas.csv: 800 filas · columnas: n, familia, tipo, delito_portal, toca_portal, fecha, buscar_por, guardar_como, paginas_omitidas, descargada
2_nuevas.csv: 403 filas · columnas: n, familia, sentencias_familia, tipo, delito_portal, disponibles, en_corpus, meta, faltan, descargadas
3_volver_a_bajar.csv: 56 filas · columnas: n, familia, tipo, delito_portal, toca_portal, fecha, buscar_por, guardar_como, motivo, descargada


{'fecha': '2025-09-10',
 'materia': 'penal',
 'delito_o_accion': 'robo-calificado',
 'tipo': 'penal/robo-calificado'}

## Fase 1. Procesamiento: del PDF al corpus

`construir-corpus --pdfs CARPETA` lee cada PDF **completo** (todas las páginas) y lo limpia:

1. **Recorte estructural** (`extract_and_clean_legal_text`): el texto empieza en la primera sección (VISTOS,
   RESULTANDO…, también numerada como "1.VISTOS") si está en el primer 10 % del documento, y termina en la primera
   fórmula de cierre (*Notifíquese*, *Así lo resolvió*…) **después** de los resolutivos. Antes, un "Notifíquese"
   citado de la primera instancia podía dejar solo el 1 % de la sentencia.
2. **Limpieza** (`limpiar_texto`): quita el anexo de datos testados, encabezados repetidos y etiquetas del PDF.
3. **Validación de la descarga** (`descargas_ajenas`): si el PDF tiene el nombre de una sentencia del corpus pero
   menos de la mitad de su texto (`contenido_en` < 0.5), es otra resolución; se rechaza y se conserva el texto
   anterior (`data/corpus/descargas_rechazadas.csv`).
4. **Repetidas** (`duplicados_de_texto`): la misma sentencia publicada dos veces con distinto nombre se cuenta una
   vez (`data/corpus/duplicados.csv`).

In [3]:
from copiloto_juridico.datos.procesamiento import extract_and_clean_legal_text, extract_text_from_pdf

pdfs = sorted(Path("data/raw").rglob("*.pdf"))
if pdfs:
    crudo = extract_text_from_pdf(pdfs[0])
    limpio = extract_and_clean_legal_text(crudo)
    print(f"{pdfs[0].name}: {len(crudo):,} caracteres leídos → {len(limpio):,} tras el recorte estructural")
    print(limpio[:600])
else:
    print("No hay PDFs en data/raw/ (son locales, fuera de git); el corpus procesado está en data/corpus/.")

2018-02-02-092953_penal_abuso-sexual-infantil-agravado-y-otro.pdf: 81,415 caracteres leídos → 59,905 tras el recorte estructural
v i s t o s para resolver los autos del toca penal número 8922017 formado con motivo del recurso de apelación interpuesto por el agente del ministerio público, en contra del auto que dejó sin materia la probanza ofertada de su parte, de fecha 29 veintinueve de agosto de 2017 dos mil diecisiete, pronunciado por el juez segundo de lo penal del primer partido judicial , dentro de la causa criminal número 4732014-c, seguida en con tra de no.1eliminadoelnombrecompleto1, por su probable responsabilidad penal en la comisión de los delitos de abuso sexual infantil agravado y violencia intrafamiliar, p


In [4]:
from copiloto_juridico.datos.corpus import contenido_en

a = "EL SUPREMO TRIBUNAL RESUELVE: PRIMERO. Se confirma el auto de vinculación a proceso. " * 5
b = "Cumplimiento de la ejecutoria de amparo: se deja insubsistente la resolución reclamada. " * 5
print("misma sentencia:", contenido_en(a, a), "· otra resolución:", contenido_en(a, b))

for nombre in ["descargas_rechazadas", "duplicados"]:
    p = Path(f"data/corpus/{nombre}.csv")
    print(f"{nombre}.csv: {len(pd.read_csv(p)):,} filas" if p.exists() else f"{nombre}.csv: no existe")

misma sentencia: 1.0 · otra resolución: 0.0
descargas_rechazadas.csv: 54 filas
duplicados.csv: 8 filas


In [5]:
from copiloto_juridico.datos.corpus import load_corpus
from copiloto_juridico.evaluacion.familias import cargar_familias

corpus = load_corpus()
penal = corpus[corpus["materia"] == "penal"]
familias = cargar_familias(penal["tipo"].unique().tolist())
omitidas = pd.read_csv("evaluacion/paginas_omitidas.csv")
print(f"Corpus: {len(corpus):,} sentencias; penales: {len(penal):,} de {penal['tipo'].nunique()} tipos del portal, "
      f"agrupados en {len(set(familias.values()))} familias de delito")
print(f"Sentencias penales que todavía tienen páginas omitidas: {omitidas['id'].isin(penal['id']).sum()}")
corpus.groupby("materia").size().rename("sentencias").to_frame()

Corpus: 4,614 sentencias; penales: 2,495 de 546 tipos del portal, agrupados en 24 familias de delito
Sentencias penales que todavía tienen páginas omitidas: 54


,sentencias
materia,
civil,1404
familiar,483
laboral,4
mercantil,228
penal,2495


In [6]:
penal.assign(familia=penal["tipo"].map(familias)).groupby("familia").size().sort_values(ascending=False) \
    .rename("sentencias").to_frame().head(12)

,sentencias
familia,
homicidio,334
robo,301
delitos-sexuales,248
delitos-culposos,236
violencia-familiar,223
fraude-y-abuso-de-confianza,170
privacion-de-la-libertad-y-secuestro,132
lesiones,129
otros,100


## Fase 2. Índice: búsqueda híbrida

`indexar` divide cada sentencia en fragmentos (≈ 400 tokens, por el límite de 512 del modelo de embeddings), calcula
sus embeddings con `multilingual-e5-base` y los guarda en `data/indice/` (incluido en el repositorio). Al arrancar,
la app arma con ellos una base ChromaDB local y un índice BM25 de palabras. La búsqueda combina las dos listas con
**RRF** (Reciprocal Rank Fusion): los embeddings encuentran sentencias con otras palabras para lo mismo y BM25
respeta los términos exactos (artículos, nombres de delitos).

In [7]:
manifest = json.loads(Path("data/indice/manifest.json").read_text())
{k: manifest[k] for k in ["embedding_model", "chunk_size", "chunk_overlap", "num_docs", "num_chunks", "dim"]}

{'embedding_model': 'intfloat/multilingual-e5-base',
 'chunk_size': 1500,
 'chunk_overlap': 200,
 'num_docs': 4614,
 'num_chunks': 211572,
 'dim': 768}

In [8]:
from copiloto_juridico.clasificador import construir_clasificador
from copiloto_juridico.vectorstore import SentenciasStore

store = SentenciasStore(log=lambda _m: None)          # la primera vez arma ChromaDB (unos minutos)
clasificador = construir_clasificador(store.corpus["tipo"].unique().tolist())
print(f"Índice listo: {len(store.corpus):,} sentencias")

/home/nacho/Documents/repos/Proyecto_Final_CopilotoJurdicoHibridoconlocalLLMyRAG/Co-piloto-Jur-dico-H-brido-con-local-LLM-y-RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16881.39it/s]

GPU sin memoria para el modelo de embeddings; se usa la CPU.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14443.85it/s]

Índice listo: 4,614 sentencias


## Fase 3. El co-piloto: los 5 pasos

El usuario describe el caso y, **si quiere**, elige uno o varios delitos. La app le sugiere los que deduce de la
descripción (sin llamar al LLM, en menos de 2 s); puede usarlos, cambiarlos o no elegir ninguno. Si elige, esas
familias sustituyen a las 3 categorías del clasificador en el filtro de la búsqueda y en el control QC_2; la búsqueda
sin filtro sigue completando los candidatos, y si la elección no coincide con lo que la app habría buscado, lo dice.

In [9]:
from copiloto_juridico.pipeline import DESCRIPCION_PASOS, PASOS

for n, (clave, titulo) in enumerate(PASOS, 1):
    print(f"{n}. {titulo}: {DESCRIPCION_PASOS[clave]}\n")

1. Extracción de conceptos clave: Entra la consulta; sale una lista de 1 a 3 conceptos jurídicos. El LLM actúa como extractor; si falla, se usa la consulta completa.

2. Clasificación híbrida de la consulta: Entran la consulta y sus conceptos; salen las 10 categorías más probables. Combina coseno de embeddings, coincidencia de lemas (Jaccard, KEM) y el voto de las sentencias recuperadas.

3. Búsqueda adaptativa de precedentes: Entran las 3 mejores categorías, o los delitos que eligió el usuario; salen hasta 30 sentencias candidatas. Fusiona búsqueda semántica (embeddings) y léxica (BM25) con RRF; si el filtro da pocas, completa sin filtro.

4. Análisis de precedentes (verificación, extracción e informe): Por candidato: QC_1 (¿es relevante?), extracción de hechos, ficha e QC_2 (¿la ficha es coherente?), hasta tener N fichas válidas. El LLM lee la sentencia recortada a su ventana de contexto.

5. Síntesis comparativa final: Entran las fichas válidas. Con reglas: qué precedentes fueron fa

In [10]:
from copiloto_juridico.ejemplos import EJEMPLOS_PENALES
from copiloto_juridico.evaluacion.familias import etiqueta_familia
from copiloto_juridico.pipeline import sugerir_familias

ejemplo = EJEMPLOS_PENALES[0]
print(ejemplo.texto, "\n")
sugerencia = sugerir_familias(store, clasificador, ejemplo.texto)
print("Sugerencia inicial de delito:", ", ".join(map(etiqueta_familia, sugerencia)))
# Es solo una sugerencia: en desarrollo la familia correcta está entre las sugeridas en 73 de 100 consultas (81 con
# los conceptos del LLM); este ejemplo es uno de los que falla (evaluacion/errores/sugerencia_delito.md).

Mi defendido fue vinculado a proceso porque supuestamente le arrebató el celular a una mujer en una parada de camión, amagándola con un cuchillo. La única prueba es el señalamiento de la víctima en un reconocimiento por fotografía hecho sin presencia de defensor. Quiero apelar el auto de vinculación. 



Sugerencia inicial de delito: delitos sexuales, feminicidio, homicidio


In [11]:
from copiloto_juridico.llm import get_client
from copiloto_juridico.pipeline import Copiloto, Opciones

cliente = get_client(llm)
ok, mensaje = cliente.health()
resultado = None
if not ok:
    print(f"Sin LLM ({mensaje}): esta celda se salta.")
else:
    # Sin elegir delito: la app lo deduce con los conceptos del LLM. Para elegir uno o varios:
    # Opciones(familias=("robo",))
    for ev in Copiloto(store, clasificador, cliente).analizar(ejemplo.texto, Opciones()):
        if ev.tipo == "paso_fin":
            print(f"✓ {ev.paso}: {ev.mensaje or ''}")
        elif ev.tipo == "error":
            print(f"✗ {ev.paso}: {ev.mensaje}")
        elif ev.tipo == "resultado":
            resultado = ev.datos["resultado"]

✓ entidades: robo calificado, vinculación a proceso, falta de contradicción en el reconocimiento por fotografía


✓ clasificacion: PENAL · familia: robo
✓ busqueda: 30 candidatos · Filtro estricto


✓ analisis: 3 informes válidos · 0 descartados


✓ sintesis: 


In [12]:
from IPython.display import Markdown

Markdown(resultado.markdown()) if resultado else print("No hay resultado (la celda anterior no corrió el análisis).")

# Informe del Co-piloto Jurídico

**Consulta:** Mi defendido fue vinculado a proceso porque supuestamente le arrebató el celular a una mujer en una parada de camión, amagándola con un cuchillo. La única prueba es el señalamiento de la víctima en un reconocimiento por fotografía hecho sin presencia de defensor. Quiero apelar el auto de vinculación.

**Categoría detectada:** PENAL · familia: robo  
**Búsqueda:** Filtro estricto  
**Modelo:** llama3.1:8b · **Tiempo:** 113 s · **Veredictos ambiguos:** 0

---
## Síntesis de precedentes relevantes

### Precedentes a favor y en contra
- **Precedente 1:** apeló agente del ministerio público contra auto de Vinculación a Proceso y Auto de No Vinculación a Proceso; la Sala MODIFICA → **parcialmente desfavorable para la defensa**.
- **Precedente 2:** apeló defensa del imputado y ministerio público contra auto de vinculación a proceso; la Sala CONFIRMA → resultado para la defensa no determinado.
- **Precedente 3:** apeló defensor oficial de la imputada contra auto de vinculación a proceso; la Sala REVOCA → **favorable para la defensa**.

### Respuesta breve
La Sala Penal del Estado de Jalisco ha revocado autos de vinculación a proceso cuando no existían pruebas suficientes para vincular a los imputados a los hechos delictivos, como en el caso de la imputada en [P3]. Además, se ha confirmado el auto de vinculación a proceso cuando las pruebas presentadas por el Ministerio Público eran suficientes para acreditar la existencia de la indagatoria y la detención del imputado, como en [P2]. Por otro lado, también se ha modificado una decisión judicial original al considerar que no se encontraban reunidos los requisitos para emitir un auto de vinculación a proceso, como en [P1].

### Argumentos que la Sala aceptó o rechazó
*   La defensa argumentó que no existen pruebas suficientes para vincular a la imputada a los hechos. **La Sala lo aceptó** porque se analizaron los antecedentes de la investigación y no se encontraron datos de prueba que corroboren la participación de la imputada en los hechos delictivos, según [P3].
*   La defensa argumentó que el auto de vinculación a proceso fue emitido sin cumplir con los requisitos establecidos por la ley. **La Sala lo aceptó** porque se consideró que no se encontraban reunidos los requisitos para emitir un auto de vinculación a proceso, según [P3].
*   La defensa argumentó que el auto de vinculación a proceso fue emitido sin cumplir con los requisitos establecidos por la ley. **La Sala lo rechazó** porque se consideró que los datos de prueba invocados por el Ministerio Público eran suficientes para acreditar la existencia de la indagatoria y la detención del imputado, según [P2].
*   La defensa argumentó que no existen pruebas suficientes para vincular a los justiciables a los hechos. **La Sala lo rechazó** porque se decreta auto de vinculación a proceso en contra de los justiciables por el delito de robo calificado, según [P1].

### Qué revisar en las sentencias
Es recomendable revisar la decisión completa de [P3] para comprender cómo se analizaron los antecedentes de la investigación y se consideró que no existían pruebas suficientes para vincular a la imputada a los hechos delictivos. Además, es conveniente revisar la fundamentación legal de [P2] para entender por qué se confirmó el auto de vinculación a proceso al considerar que las pruebas presentadas por el Ministerio Público eran suficientes para acreditar la existencia de la indagatoria y la detención del imputado.

---
## Apéndice: análisis detallado de precedentes

### Precedente 1: PENAL · robo calificado y tentativa de robo calificado

| Documento | Fecha | Materia | Similitud | Fuente |
|---|---|---|---|---|
| `2024-02-29-121709_penal_robo-calificado-y-tentativa-de-robo-calificado.pdf` | 2024-02-29 | PENAL | 0.877 | Filtro estricto |

#### Resumen
El recurso de apelación interpuesto por el Agente del Ministerio Público se resolvió modificando la decisión judicial original. Se decreta auto de vinculación a proceso en contra de [eliminados] por el delito de robo calificado y se sostiene el auto de no vinculación a proceso, a favor de los justiciables [eliminados].

#### Datos del proceso
- **Tribunal:** Sala Penal del Estado de Jalisco · 29 de noviembre de 2023
- **Toca / expediente:** Toca / Expediente / Causa 378/2023
- **Acto impugnado:** Auto de Vinculación a Proceso y Auto de No Vinculación a Proceso · **Apelante:** Agente del Ministerio Público
- **Delito(s):** Delito de Robo Calificado (Art. 233 en relación con el Art. 236 fracción xi) y Delito de Robo Calificado en Grado de Tentativa (Art. 233 en relación con el Art. 236 fracciones ix, xi, xii y xix)
- **Sentido del fallo:** MODIFICA
- **Resultado para la defensa:** parcialmente desfavorable

#### Agravios y su calificación
- [DATO NO DISPONIBLE]

#### Hechos y decisión
**Hechos:** El juez de control determinó auto de vinculación a proceso a [eliminados] por el delito de robo calificado y auto de no vinculación a proceso a otros [eliminados] por el delito de robo calificado en grado de tentativa.

**Decisión:** Se decreta auto de vinculación a proceso en contra de [eliminados] por el delito de robo calificado y se sostiene el auto de no vinculación a proceso, a favor de los justiciables [eliminados], al no tenerse por acreditado el hecho que la ley señala como delito de robo calificado en grado de tentativa.

#### Fundamento legal
- Art. 316 y 318 del Código Nacional de Procedimientos Penales
- Art. 233 y 236 fracción xi del Código Penal para el Estado de Jalisco

#### Pruebas decisivas
- Registros de audio y video de la audiencia de fecha 05 de julio de 2023

### Precedente 2: PENAL · robo calificado y privacion ilegal de la libertad y otros derechos

| Documento | Fecha | Materia | Similitud | Fuente |
|---|---|---|---|---|
| `2023-06-14-221053_penal_robo-calificado-y-privacion-ilegal-de-la-libertad-y-otros-derechos.pdf` | 2023-06-14 | PENAL | 0.869 | Filtro estricto |

#### Resumen
El Tribunal confirmó el auto de vinculación a proceso dictado en contra del imputado, por los delitos de robo calificado y privación ilegal de la libertad, cometidos en agravio de [eliminado] y [eliminado], así como la persona moral [eliminado] y [eliminado].

#### Datos del proceso
- **Tribunal:** Sala Penal del Primer Circuito Judicial del Estado de Jalisco · 29 de marzo de 2023
- **Toca / expediente:** Toca número indicado en el encabezado
- **Acto impugnado:** Auto de vinculación a proceso · **Apelante:** Defensa del imputado y Ministerio Público
- **Delito(s):** Robo calificado y privación ilegal de la libertad, previstos por el artículo 233 en relación al 236 fracciones ix, xi y xii y 193, en contexto al 14 fracción i, 15 y 19 fracción iii, todos del código penal del estado de Jalisco
- **Sentido del fallo:** CONFIRMA
- **Resultado para la defensa:** [DATO NO DISPONIBLE]

#### Agravios y su calificación
- Se pone en duda la existencia de datos de prueba que permitan establecer la probabilidad de que el imputado haya cometido los hechos o participado en su comisión. — **INFUNDADO**: La Sala consideró que los datos de prueba invocados por el Ministerio Público eran suficientes para acreditar la existencia de la indagatoria y la detención del imputado.

#### Hechos y decisión
**Hechos:** El imputado fue sometido a un robo calificado y privación ilegal de la libertad el día 19 de diciembre de 2019. Las víctimas fueron apuntadas con una arma de fuego y obligadas a bajarse del vehículo.

**Decisión:** Se confirma el auto de vinculación a proceso dictado en la audiencia de fecha 26 de mayo de 2022, por el juez décimo cuarto especializado en control, enjuiciamiento y justicia integral para adolescentes, del sistema penal acusatorio adversaria, del primer partido judicial del estado de Jalisco.

#### Fundamento legal
- Artículo 16 Constitucional
- Artículos 478 y 479 del Código Nacional de Procedimientos Penales
- Artículo 47 Fracción I de la Ley Orgánica del Poder Judicial del Estado de Jalisco

#### Pruebas decisivas
- La indagatoria y la detención del imputado
- Los datos de prueba invocados por el Ministerio Público
- Las declaraciones de las víctimas

### Precedente 3: PENAL · robo calificado y otro

| Documento | Fecha | Materia | Similitud | Fuente |
|---|---|---|---|---|
| `2025-11-05-140542_penal_robo-calificado-y-otro.pdf` | 2025-11-05 | PENAL | 0.865 | Filtro estricto |

#### Resumen
Se revoca el auto de vinculación a proceso y se dicta auto de no vinculación a proceso a favor de la imputada, al no existir pruebas suficientes para vincularla a los hechos delictivos.

#### Datos del proceso
- **Tribunal:** Sala Penal · 29/12/2024
- **Toca / expediente:** Toca penal de juicio oral número 2052025
- **Acto impugnado:** Auto de vinculación a proceso · **Apelante:** Defensor oficial de la imputada
- **Delito(s):** Robo calificado y extorsión, previstos y sancionados por los artículos 233 en relación al 236 fracciones vii, ix, xi y xii, así como por el diverso 189 del código penal del estado de jalisco
- **Sentido del fallo:** REVOCA
- **Resultado para la defensa:** favorable

#### Agravios y su calificación
- La defensa argumentó que no existen pruebas suficientes para vincular a la imputada a los hechos. — **FUNDADO**: Porque se analizaron los antecedentes de la investigación y no se encontraron datos de prueba que corroboren la participación de la imputada en los hechos delictivos.
- La defensa argumentó que el auto de vinculación a proceso fue emitido sin cumplir con los requisitos establecidos por la ley. — **FUNDADO**: Porque se consideró que no se encontraban reunidos los requisitos para emitir un auto de vinculación a proceso.

#### Hechos y decisión
**Hechos:** La imputada denunció el robo de una camioneta, pero no se encontraron pruebas que la vinculen a los hechos. Las víctimas declararon que fueron amenazadas y que les pidieron sus teléfonos y contraseñas.

**Decisión:** Se revoca el auto de vinculación a proceso y se dicta auto de no vinculación a proceso a favor de la imputada.

#### Fundamento legal
- Artículo 316 del código nacional de procedimientos penales
- Artículo 19 constitucional

#### Pruebas decisivas
- La declaración de las víctimas, el reporte de robo presentado por la imputada


---
_Herramienta de apoyo basada en precedentes públicos del STJ Jalisco. No sustituye la asesoría de un profesional del derecho._

## Fase 4. Evaluación: cómo se midió

No hay abogados disponibles para anotar, así que la referencia es **automática y externa al sistema**:

- **Delito:** la categoría que asignó el propio STJ en su portal, agrupada en familias (`evaluacion/familias_penal.csv`).
- **Acto impugnado, apelante y sentido del fallo:** mayoría entre las reglas (`metadatos.py`) y un juez LLM de otra
  familia de modelos que la app. Un juez solo cuenta si acierta la familia del STJ en al menos **0.80** de las
  sentencias, umbral fijado antes de correrlo: `gemma2:9b` pasó (0.845); `mistral:7b` no (0.795).
- **Relevancia de un par (consulta, sentencia):** 2 = mismo delito y mismo problema procesal, 1 = uno de los dos,
  0 = ninguno.
- **Muestra estable por hash** (30 % de las sentencias, 50 % de las consultas): el mismo elemento siempre queda dentro
  o fuera aunque crezca el corpus, y no hace falta juzgar todo.
- **Congelar y medir una vez:** `congelar` guarda la huella del código, datos, prompts y modelo; cada corrida de la
  partición de prueba se niega a empezar si algo cambió y solo se permite una vez.

Cada corrida queda en `evaluacion/resultados/` con fecha, commit, configuración, semilla y huella del corpus, y cada
decisión de `config.py` cita la corrida que la respalda.

In [13]:
corridas = []
for p in sorted(Path("evaluacion/resultados").glob("*.json")):
    d = json.loads(p.read_text())
    prueba = (d.get("parametros") or {}).get("particion") == "prueba"
    corridas.append({"corrida": p.stem, "comando": d["comando"], "fecha": d["fecha"][:16], "commit": d["commit"],
                     "partición de prueba": "sí" if prueba else ""})
pd.DataFrame(corridas).groupby("comando").tail(1).set_index("comando")   # la última de cada comando

,corrida,fecha,commit,partición de prueba
comando,,,,
latencia,20261113-112440_fb1f3fe_latencia,2026-11-13T11:24,fb1f3fe,
qc-diagnostico,20261113-114412_fb1f3fe_qc-diagnostico,2026-11-13T11:44,fb1f3fe,
regresion,20261113-114442_fb1f3fe_regresion,2026-11-13T11:44,fb1f3fe,
robustez,20261113-115133_fb1f3fe_robustez,2026-11-13T11:51,fb1f3fe,
demo,20261113-120016_fb1f3fe_demo,2026-11-13T12:00,fb1f3fe,
modelos,20261113-125326_fb1f3fe_modelos,2026-11-13T12:53,fb1f3fe,
recuperacion,20261113-130208_fb1f3fe_recuperacion,2026-11-13T13:02,fb1f3fe,
referencias,20261117-065346_a60ca4f_referencias,2026-11-17T06:53,a60ca4f,
fichas-diagnostico,20261119-224632_96142b0_fichas-diagnostico,2026-11-19T22:46,96142b0,


In [14]:
from copiloto_juridico.evaluacion.reporte import ultimo_resultado

ref = ultimo_resultado("referencias")
t = ref["tablas"]["Validación de los jueces"]
print(ref["_archivo"], "·", t["descripcion"])
pd.DataFrame(t["filas"], columns=t["columnas"])

20261117-065346_a60ca4f_referencias.json · Exactitud de la familia del delito que lee cada juez contra la categoría que asignó el STJ (referencia externa; una respuesta inválida cuenta como error). Un juez se usa si alcanza el mínimo, fijado antes de correrlos.


,juez,sentencias,con respuesta válida,exactitud familia (STJ),IC95,mínimo,se usa
0,mistral:7b,1412,1407,0.795326,"[0.773, 0.816]",0.8,no
1,gemma2:9b,401,401,0.845387,"[0.807, 0.877]",0.8,sí


## Limitaciones

- La referencia es lo que dicen el tribunal (su categoría) y el texto de la sentencia, leído por reglas y por un
  juez LLM; no el juicio de un abogado sobre la utilidad de un precedente.
- Un solo juez LLM pasó el umbral, y juzga una muestra.
- Algunas sentencias penales siguen con páginas omitidas (la descarga a mano trajo otra resolución); están en
  `evaluacion/paginas_omitidas.csv`.
- Sin OCR: los PDF escaneados no aportan texto.
- El aviso de consulta fuera de la materia penal se basa en una lista de palabras.

**Aviso legal:** herramienta académica basada en sentencias públicas. No constituye asesoría jurídica; las fichas las
genera un modelo de lenguaje y pueden tener errores: verifica siempre la sentencia original.